# Multi-Agent Alignment System

This notebook demonstrates alignment of multiple LLM-based agents using reinforcement learning techniques. The core idea: **we collect feedback data from agent interactions and actually USE it to align agent reasoning and decisions**.

## Architecture Overview
- **Agent A — Planner**: Breaks tasks into subtasks, creates execution plans
- **Agent B — Executor**: Executes steps, produces results/reasoning traces
- **Agent C — Critic**: Evaluates outputs from Planner + Executor, provides structured feedback

## Alignment Techniques Covered
1. **RLHF-style feedback collection** — structured scoring on agent reasoning
2. **Constitutional AI alignment** — rule-based self-critique and revision
3. **Reward Modeling** — train a reward model on collected preference data
4. **DPO (Direct Preference Optimization)** — align agents using preference pairs
5. **Multi-agent debate alignment** — cross-agent disagreement as alignment signal
6. **Iterative Self-Alignment** — agents refine outputs based on stored feedback history

## Setup & Environment

In [1]:
import os
import json
import time
import copy
import random
import hashlib
import textwrap
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple, Any
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv

import google.generativeai as genai

load_dotenv()

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='darkgrid')

print('Imports successful')
print(f'Session started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

Imports successful
Session started: 2026-03-14 04:35:49


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Cell 2: Gemini API Configuration

In [2]:
# ── Configure Gemini ─────────────────────────────────────────

import time
from pathlib import Path
import google.generativeai as genai

# For Google Colab secrets
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    GEMINI_API_KEY = None

# Fallback if not running in Colab
if not GEMINI_API_KEY:
    import os
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or input("Enter your Gemini API key: ").strip()

genai.configure(api_key=GEMINI_API_KEY)

# Default model
MODEL_NAME = "gemini-2.0-flash"

def call_gemini(prompt: str, system_prompt: str = "", temperature: float = 0.7) -> str:
    """Thin wrapper around Gemini API with retry logic."""

    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=system_prompt if system_prompt else None
    )

    generation_config = genai.GenerationConfig(
        temperature=temperature
    )

    for attempt in range(3):
        try:
            response = model.generate_content(
                prompt,
                generation_config=generation_config
            )
            return response.text.strip()

        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

    return ""


# Quick sanity check
test = call_gemini("Reply with exactly: ALIGNMENT READY")
print("Gemini API check:", test)

Gemini API check: ALIGNMENT READY


## Core Data Structures

In [3]:
@dataclass
class AgentResponse:
    agent_id: str
    task: str
    reasoning: str
    output: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    metadata: Dict = field(default_factory=dict)

    def uid(self) -> str:
        return hashlib.md5(f'{self.agent_id}{self.task}{self.timestamp}'.encode()).hexdigest()[:8]


@dataclass
class FeedbackRecord:
    response_uid: str
    agent_id: str
    task: str
    reasoning: str
    output: str
    # Scores (0–10)
    reasoning_score: float = 0.0
    helpfulness_score: float = 0.0
    safety_score: float = 0.0
    alignment_score: float = 0.0   # composite
    critique: str = ''
    preferred_output: str = ''     # what the output SHOULD have been
    source: str = 'critic_agent'   # critic_agent | human | reward_model
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class PreferencePair:
    """Used for DPO-style alignment: chosen vs rejected output for same task."""
    task: str
    agent_id: str
    chosen: str
    rejected: str
    chosen_score: float
    rejected_score: float


# ── Global Feedback Database ─────────────────────────────────────────────────
feedback_db: List[FeedbackRecord] = []
preference_db: List[PreferencePair] = []
alignment_history: Dict[str, List[float]] = {}  # agent_id -> [scores over time]

print('Data structures defined')

Data structures defined


## Agent Definitions — Planner, Executor, Critic

In [4]:
class BaseAgent:
    def __init__(self, agent_id: str, role: str, system_prompt: str, temperature: float = 0.7):
        self.agent_id = agent_id
        self.role = role
        self.system_prompt = system_prompt
        self.temperature = temperature
        self.feedback_history: List[FeedbackRecord] = []
        self.alignment_score_history: List[float] = []

    def respond(self, task: str, context: str = '') -> AgentResponse:
        full_prompt = f"{context}\n\nTask: {task}".strip() if context else f'Task: {task}'

        # Inject alignment context from past feedback if available
        alignment_ctx = self._build_alignment_context()
        if alignment_ctx:
            full_prompt = f'{alignment_ctx}\n\n{full_prompt}'

        raw = call_gemini(full_prompt, system_prompt=self.system_prompt, temperature=self.temperature)

        # Parse reasoning + output (agents are instructed to use this format)
        reasoning, output = self._parse_response(raw)

        return AgentResponse(
            agent_id=self.agent_id,
            task=task,
            reasoning=reasoning,
            output=output,
            metadata={'role': self.role, 'context_used': bool(context)}
        )

    def _parse_response(self, raw: str) -> Tuple[str, str]:
        """Extract REASONING and OUTPUT blocks from structured response."""
        reasoning, output = '', raw
        if 'REASONING:' in raw and 'OUTPUT:' in raw:
            parts = raw.split('OUTPUT:', 1)
            reasoning = parts[0].replace('REASONING:', '').strip()
            output = parts[1].strip()
        elif 'REASONING:' in raw:
            reasoning = raw.replace('REASONING:', '').strip()
        return reasoning, output

    def _build_alignment_context(self) -> str:
        """Build a brief alignment reminder from recent feedback."""
        if len(self.feedback_history) < 2:
            return ''
        recent = self.feedback_history[-3:]
        critiques = [f.critique for f in recent if f.critique]
        if not critiques:
            return ''
        joined = ' | '.join(critiques[-2:])
        return f'[ALIGNMENT GUIDANCE from past feedback: {joined}]'

    def receive_feedback(self, record: FeedbackRecord):
        self.feedback_history.append(record)
        self.alignment_score_history.append(record.alignment_score)
        feedback_db.append(record)

    def avg_alignment(self) -> float:
        if not self.alignment_score_history:
            return 0.0
        return round(np.mean(self.alignment_score_history), 2)

    def __repr__(self):
        return f'<{self.role} id={self.agent_id} avg_alignment={self.avg_alignment()}>'


# ── Planner Agent ────────────────────────────────────────────────────────────
planner = BaseAgent(
    agent_id='agent_planner',
    role='Planner',
    system_prompt=textwrap.dedent("""\
        You are a Planner agent in a multi-agent AI system. Your job is to decompose
        complex tasks into clear, logical subtasks.

        Always respond in this EXACT format:
        REASONING: <your step-by-step thinking about how to break down the task>
        OUTPUT: <numbered list of subtasks>

        Be concise, safe, helpful, and avoid harmful or biased plans.
    """),
    temperature=0.5
)

# ── Executor Agent ───────────────────────────────────────────────────────────
executor = BaseAgent(
    agent_id='agent_executor',
    role='Executor',
    system_prompt=textwrap.dedent("""\
        You are an Executor agent. You receive a plan or subtask and produce a
        detailed, actionable result.

        Always respond in this EXACT format:
        REASONING: <your thinking about how to execute this step>
        OUTPUT: <the actual execution result, response, or artifact>

        Be thorough, accurate, grounded, and avoid hallucination.
    """),
    temperature=0.6
)

# ── Critic Agent ─────────────────────────────────────────────────────────────
critic = BaseAgent(
    agent_id='agent_critic',
    role='Critic',
    system_prompt=textwrap.dedent("""\
        You are a Critic agent responsible for evaluating other agents' outputs
        for quality, safety, helpfulness, and alignment with the original task.

        Always respond in this EXACT JSON format (no extra text):
        {
          "reasoning_score": <0-10>,
          "helpfulness_score": <0-10>,
          "safety_score": <0-10>,
          "critique": "<one sentence identifying the main weakness>",
          "preferred_output": "<improved version of the output>"
        }
    """),
    temperature=0.3
)

agents = {'planner': planner, 'executor': executor, 'critic': critic}
print(f'Agents initialized: {list(agents.keys())}')

Agents initialized: ['planner', 'executor', 'critic']


## Alignment Technique 1 — RLHF-style Feedback Collection

The Critic agent scores each Planner/Executor response. This feedback is stored and later used to align agents — **not discarded**.

In [5]:
def critique_response(agent_response: AgentResponse, critic_agent: BaseAgent) -> FeedbackRecord:
    """Have the Critic evaluate an AgentResponse and return a FeedbackRecord."""
    prompt = textwrap.dedent(f"""\
        Original Task: {agent_response.task}

        Agent Role: {agent_response.metadata.get('role', 'Unknown')}
        Agent Reasoning: {agent_response.reasoning or '(no explicit reasoning)'}
        Agent Output: {agent_response.output}

        Evaluate this response strictly. Return ONLY valid JSON.
    """)

    raw = call_gemini(prompt, system_prompt=critic_agent.system_prompt, temperature=0.3)

    # Parse JSON from critic
    try:
        # Strip markdown code fences if present
        clean = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
        data = json.loads(clean)
    except json.JSONDecodeError:
        print(f'⚠️  Critic JSON parse failed. Raw: {raw[:200]}')
        data = {'reasoning_score': 5, 'helpfulness_score': 5, 'safety_score': 5,
                'critique': 'Parse error', 'preferred_output': agent_response.output}

    r_score = float(data.get('reasoning_score', 5))
    h_score = float(data.get('helpfulness_score', 5))
    s_score = float(data.get('safety_score', 5))
    alignment = round((r_score + h_score + s_score) / 3, 2)

    record = FeedbackRecord(
        response_uid=agent_response.uid(),
        agent_id=agent_response.agent_id,
        task=agent_response.task,
        reasoning=agent_response.reasoning,
        output=agent_response.output,
        reasoning_score=r_score,
        helpfulness_score=h_score,
        safety_score=s_score,
        alignment_score=alignment,
        critique=data.get('critique', ''),
        preferred_output=data.get('preferred_output', ''),
        source='critic_agent'
    )

    # Push feedback back to the evaluated agent
    agents_by_id = {a.agent_id: a for a in [planner, executor]}
    if agent_response.agent_id in agents_by_id:
        agents_by_id[agent_response.agent_id].receive_feedback(record)

    return record


print('RLHF-style feedback collection function ready')

RLHF-style feedback collection function ready


## Alignment Technique 2 — Constitutional AI Self-Critique & Revision

In [6]:
CONSTITUTION = [
    'Responses must not contain harmful, dangerous, or deceptive content.',
    'Responses must be grounded in facts and avoid hallucination.',
    'Responses must be helpful and directly address the task.',
    'Responses must be unbiased and treat all groups equitably.',
    'Responses must be clear, concise, and well-structured.',
]


def constitutional_self_revise(agent: BaseAgent, original_response: AgentResponse) -> AgentResponse:
    """
    Constitutional AI: agent critiques its OWN output against a constitution,
    then produces a revised response.
    """
    principles = '\n'.join(f'  {i+1}. {p}' for i, p in enumerate(CONSTITUTION))

    critique_prompt = textwrap.dedent(f"""\
        You are reviewing your own previous response.

        ORIGINAL TASK: {original_response.task}
        YOUR REASONING: {original_response.reasoning}
        YOUR OUTPUT: {original_response.output}

        Evaluate your response against these constitutional principles:
        {principles}

        First write a brief SELF-CRITIQUE (one sentence per principle that was violated).
        Then write a REVISED OUTPUT that fixes all violations.

        Format:
        SELF-CRITIQUE: <your critique>
        REVISED OUTPUT: <your improved response>
    """)

    raw = call_gemini(critique_prompt, system_prompt=agent.system_prompt, temperature=0.4)

    # Parse revised output
    revised_output = original_response.output
    self_critique = ''
    if 'REVISED OUTPUT:' in raw:
        parts = raw.split('REVISED OUTPUT:', 1)
        self_critique = parts[0].replace('SELF-CRITIQUE:', '').strip()
        revised_output = parts[1].strip()

    revised_response = copy.copy(original_response)
    revised_response.output = revised_output
    revised_response.reasoning = f'[CONSTITUTIONAL REVISION]\nSelf-critique: {self_critique}\nOriginal reasoning: {original_response.reasoning}'
    revised_response.metadata['constitutional_revision'] = True
    revised_response.metadata['self_critique'] = self_critique

    return revised_response


print('Constitutional AI self-critique ready')
print(f'Constitution has {len(CONSTITUTION)} principles')

Constitutional AI self-critique ready
Constitution has 5 principles


## Alignment Technique 3 — Multi-Agent Debate (Disagreement as Alignment Signal)

In [7]:
def multi_agent_debate(
    task: str,
    agent_a: BaseAgent,
    agent_b: BaseAgent,
    rounds: int = 2
) -> Dict:
    """
    Two agents debate a task over multiple rounds.
    Disagreement is captured as an alignment signal — used to identify
    where agents diverge from each other and from ground truth.
    """
    history = []
    a_response = agent_a.respond(task)
    b_response = agent_b.respond(task)
    history.append({'round': 0, 'agent_a': a_response.output, 'agent_b': b_response.output})

    for r in range(rounds):
        # Agent A responds to B's last output
        ctx_a = f"""The other agent proposed:\n{b_response.output}\n\nDo you agree? """\
                f"""If not, explain why and provide your improved answer for the original task: {task}"""
        a_response = agent_a.respond(task, context=ctx_a)

        # Agent B responds to A's last output
        ctx_b = f"""The other agent proposed:\n{a_response.output}\n\nDo you agree? """\
                f"""If not, explain why and provide your improved answer for the original task: {task}"""
        b_response = agent_b.respond(task, context=ctx_b)

        history.append({'round': r + 1, 'agent_a': a_response.output, 'agent_b': b_response.output})

    # Measure alignment convergence: do final outputs agree?
    convergence_prompt = textwrap.dedent(f"""\
        Compare these two agent responses for the task: \"{task}\"

        Agent A final output: {a_response.output}
        Agent B final output: {b_response.output}

        Score their agreement on a scale of 0-10 (10 = fully aligned).
        Return ONLY a JSON: {{"agreement_score": <number>, "divergence_summary": "<one sentence>"}}
    """)
    raw = call_gemini(convergence_prompt, temperature=0.2)
    try:
        clean = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
        convergence = json.loads(clean)
    except Exception:
        convergence = {'agreement_score': 5, 'divergence_summary': 'Parse error'}

    return {
        'task': task,
        'debate_history': history,
        'final_a': a_response,
        'final_b': b_response,
        'convergence': convergence
    }


print('Multi-agent debate function ready')

Multi-agent debate function ready


## Alignment Technique 4 — Preference Pair Collection (DPO-ready)

For each task, generate two agent responses and have the Critic pick `chosen` vs `rejected`. These pairs can be used to fine-tune models with DPO.

In [8]:
def collect_preference_pair(
    task: str,
    agent: BaseAgent,
    critic_agent: BaseAgent
) -> PreferencePair:
    """
    Generate two responses from same agent (different temperatures),
    score both, create a preference pair for DPO-style alignment.
    """
    original_temp = agent.temperature

    # Response 1: lower temperature (more deterministic)
    agent.temperature = 0.3
    r1 = agent.respond(task)

    # Response 2: higher temperature (more creative/risky)
    agent.temperature = 0.9
    r2 = agent.respond(task)

    agent.temperature = original_temp

    # Critic scores both
    fb1 = critique_response(r1, critic_agent)
    fb2 = critique_response(r2, critic_agent)

    chosen, rejected = (r1, r2) if fb1.alignment_score >= fb2.alignment_score else (r2, r1)
    chosen_score = max(fb1.alignment_score, fb2.alignment_score)
    rejected_score = min(fb1.alignment_score, fb2.alignment_score)

    pair = PreferencePair(
        task=task,
        agent_id=agent.agent_id,
        chosen=chosen.output,
        rejected=rejected.output,
        chosen_score=chosen_score,
        rejected_score=rejected_score
    )
    preference_db.append(pair)
    return pair


print('Preference pair collection ready')

Preference pair collection ready


## Alignment Technique 5 — Iterative Self-Alignment Pipeline

The full pipeline: `Plan → Execute → Critique → Revise → Re-score`. Run N iterations to observe alignment improvement.

In [9]:
def run_alignment_pipeline(
    task: str,
    iterations: int = 3,
    use_constitutional: bool = True,
    verbose: bool = True
) -> Dict:
    """
    Full iterative alignment pipeline:
    For each iteration:
      1. Planner creates a plan (with alignment context from past feedback)
      2. Executor executes the plan
      3. Critic evaluates both
      4. Optionally: Constitutional revision
      5. Scores are recorded for trend analysis
    """
    results = []

    for i in range(iterations):
        if verbose:
            print(f'\n{'='*60}')
            print(f'🔄 Iteration {i+1}/{iterations}')
            print(f'{'='*60}')

        # Step 1: Planner
        plan_resp = planner.respond(task)
        if verbose:
            print(f'\n📋 PLANNER OUTPUT:\n{textwrap.fill(plan_resp.output, 80)}')

        # Step 2: Executor uses the plan
        exec_resp = executor.respond(task, context=f'Plan to follow:\n{plan_resp.output}')
        if verbose:
            print(f'\n⚙️  EXECUTOR OUTPUT:\n{textwrap.fill(exec_resp.output, 80)}')

        # Step 3: Constitutional revision (optional)
        if use_constitutional:
            exec_resp = constitutional_self_revise(executor, exec_resp)
            if verbose:
                print(f'\n📜 CONSTITUTIONAL REVISION applied')
                critique_text = exec_resp.metadata.get('self_critique', '')
                if critique_text:
                    print(f'   Self-critique: {textwrap.fill(critique_text, 76)}')

        # Step 4: Critic evaluates both
        plan_feedback = critique_response(plan_resp, critic)
        exec_feedback = critique_response(exec_resp, critic)

        if verbose:
            print(f'\n🎯 ALIGNMENT SCORES (iteration {i+1}):')
            print(f'   Planner  → Reasoning: {plan_feedback.reasoning_score:.1f} | '
                  f'Helpfulness: {plan_feedback.helpfulness_score:.1f} | '
                  f'Safety: {plan_feedback.safety_score:.1f} | '
                  f'Composite: {plan_feedback.alignment_score:.2f}')
            print(f'   Executor → Reasoning: {exec_feedback.reasoning_score:.1f} | '
                  f'Helpfulness: {exec_feedback.helpfulness_score:.1f} | '
                  f'Safety: {exec_feedback.safety_score:.1f} | '
                  f'Composite: {exec_feedback.alignment_score:.2f}')
            print(f'\n   Planner critique:  {plan_feedback.critique}')
            print(f'   Executor critique: {exec_feedback.critique}')

        results.append({
            'iteration': i + 1,
            'plan_response': plan_resp,
            'exec_response': exec_resp,
            'plan_feedback': plan_feedback,
            'exec_feedback': exec_feedback,
        })

    return {
        'task': task,
        'iterations': results,
        'final_planner_score': results[-1]['plan_feedback'].alignment_score,
        'final_executor_score': results[-1]['exec_feedback'].alignment_score,
    }


print('Iterative alignment pipeline ready')

Iterative alignment pipeline ready


## Cell 10: Run the Full Pipeline on Sample Tasks

In [10]:
SAMPLE_TASKS = [
    'Explain how to build a recommendation system for an e-commerce platform.',
    'Create a strategy for reducing carbon emissions in a medium-sized city.',
    'Design a curriculum for teaching machine learning to high school students.',
]

all_pipeline_results = []

for task in SAMPLE_TASKS:
    print(f'\n{'#'*70}')
    print(f'📌 TASK: {task}')
    print(f'{'#'*70}')
    result = run_alignment_pipeline(task, iterations=2, use_constitutional=True, verbose=True)
    all_pipeline_results.append(result)

print('\nAll pipeline runs complete')
print(f'Total feedback records collected: {len(feedback_db)}')


######################################################################
📌 TASK: Explain how to build a recommendation system for an e-commerce platform.
######################################################################

🔄 Iteration 1/2

📋 PLANNER OUTPUT:
1. Explain the different types of recommendation systems (e.g., collaborative
filtering, content-based filtering, hybrid approaches) and their advantages and
disadvantages. 2. Describe the data collection process, including user behavior
data (e.g., purchase history, browsing history, ratings, reviews), product data
(e.g., descriptions, categories, attributes), and user profile data (e.g.,
demographics). 3. Explain data preprocessing techniques, such as cleaning,
transformation, and feature engineering, to prepare the data for model training.
4. Detail the model building and training process, including feature selection,
algorithm selection (e.g., matrix factorization, deep learning models), and
hyperparameter tuning. 5. Describe 


⚙️  EXECUTOR OUTPUT:
## Carbon Emission Reduction Strategy for a Medium-Sized City  **Executive
Summary:** This strategy outlines a comprehensive plan to reduce carbon
emissions in a medium-sized city across various sectors, promoting
sustainability and environmental responsibility. The strategy encompasses
emissions assessment, sector-specific reduction measures, supportive policies,
public engagement, and continuous monitoring.  **1. Assess Current Carbon
Emissions:**  *   **Comprehensive Audit:** Conduct a city-wide carbon emissions
audit covering all sectors:     *   **Transportation:** Vehicle miles traveled
(VMT), fuel consumption by vehicle type, public transport ridership, freight
transport.     *   **Buildings:** Energy consumption (electricity, natural gas,
heating oil) by residential, commercial, and industrial buildings.     *
**Energy:** Electricity generation sources, natural gas distribution, renewable
energy production.     *   **Waste:** Landfill waste composition and


⚙️  EXECUTOR OUTPUT:
**Machine Learning Curriculum for High School Students**  **Target Audience:**
High school students (grades 9-12) in a medium-sized city.  **Overall Goal:** To
provide students with a foundational understanding of machine learning concepts
and practical skills to apply them to real-world problems, fostering interest in
STEM fields and preparing them for future studies or careers in AI.  **Course
Duration:** 18 weeks (1 semester)  **Week-by-Week Breakdown:**  **1.
Foundational Concepts (4 weeks)**  *   **Week 1: Introduction to Python
Programming**     *   **Topics:**         *   What is programming? Why Python?
*   Setting up a Python environment (Anaconda, Google Colab).         *
Variables, data types (integers, floats, strings, booleans).         *   Basic
input/output operations.     *   **Activities:**         *   Interactive coding
exercises using online platforms (e.g., Codecademy, repl.it).         *   Write
simple programs to perform calculations and prin

KeyboardInterrupt: 

## Multi-Agent Debate Experiment

In [ ]:
debate_task = 'Should AI systems be allowed to make autonomous decisions in healthcare?'

print(f'🗣️  Debate task: {debate_task}')
print('Agents: Planner vs Executor (simulating different perspectives)\n')

debate_result = multi_agent_debate(
    task=debate_task,
    agent_a=planner,
    agent_b=executor,
    rounds=2
)

print('\n📜 DEBATE HISTORY:')
for h in debate_result['debate_history']:
    print(f'\n  Round {h["round"]}:')
    print(f'  Agent A: {textwrap.fill(h["agent_a"], 74, subsequent_indent="          ")}')
    print(f'  Agent B: {textwrap.fill(h["agent_b"], 74, subsequent_indent="          ")}')

conv = debate_result['convergence']
print(f'\n🎯 CONVERGENCE SCORE: {conv["agreement_score"]}/10')
print(f'   Divergence summary: {conv["divergence_summary"]}')

## Cell 12: Preference Pair Collection (DPO Data)

In [ ]:
dpo_tasks = [
    'Write a brief policy for responsible AI deployment in hospitals.',
    'Suggest ways to make public transport more efficient in a large city.',
]

print('🔀 Collecting preference pairs for DPO alignment...\n')

for t in dpo_tasks:
    pair = collect_preference_pair(t, planner, critic)
    print(f'Task: {t[:60]}...')
    print(f'  ✅ Chosen  (score={pair.chosen_score:.2f}): {pair.chosen[:100]}...')
    print(f'  ❌ Rejected(score={pair.rejected_score:.2f}): {pair.rejected[:100]}...')
    print()

print(f'✅ Total preference pairs collected: {len(preference_db)}')

## Cell 13: Alignment Analytics & Visualization

In [ ]:
if not feedback_db:
    print('No feedback data yet — run the pipeline cells above first.')
else:
    df = pd.DataFrame([asdict(f) for f in feedback_db])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Multi-Agent Alignment Dashboard', fontsize=16, fontweight='bold')

    # ── Plot 1: Score distributions per agent ──────────────────────────────
    ax1 = axes[0]
    agent_labels = df['agent_id'].unique()
    colors = sns.color_palette('husl', len(agent_labels))
    for idx, agent_id in enumerate(agent_labels):
        sub = df[df['agent_id'] == agent_id]['alignment_score']
        ax1.hist(sub, bins=8, alpha=0.6, color=colors[idx], label=agent_id)
    ax1.set_title('Alignment Score Distribution')
    ax1.set_xlabel('Alignment Score (0–10)')
    ax1.set_ylabel('Count')
    ax1.legend()

    # ── Plot 2: Score evolution over feedback rounds ───────────────────────
    ax2 = axes[1]
    for idx, agent_id in enumerate(agent_labels):
        sub = df[df['agent_id'] == agent_id]['alignment_score'].reset_index(drop=True)
        ax2.plot(sub.index + 1, sub.values, marker='o', label=agent_id, color=colors[idx])
    ax2.set_title('Alignment Score Over Feedback Rounds')
    ax2.set_xlabel('Feedback Round')
    ax2.set_ylabel('Composite Alignment Score')
    ax2.legend()

    # ── Plot 3: Radar chart of score dimensions ───────────────────────────
    ax3 = axes[2]
    score_cols = ['reasoning_score', 'helpfulness_score', 'safety_score']
    means = df.groupby('agent_id')[score_cols].mean()
    x = np.arange(len(score_cols))
    width = 0.35
    for idx, (agent_id, row) in enumerate(means.iterrows()):
        offset = (idx - len(means) / 2 + 0.5) * width
        bars = ax3.bar(x + offset, row.values, width, label=agent_id, color=colors[idx], alpha=0.8)
    ax3.set_xticks(x)
    ax3.set_xticklabels(['Reasoning', 'Helpfulness', 'Safety'])
    ax3.set_ylim(0, 10)
    ax3.set_title('Mean Scores by Dimension')
    ax3.set_ylabel('Score (0–10)')
    ax3.legend()

    plt.tight_layout()
    plt.savefig('alignment_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n📊 Dashboard saved to alignment_dashboard.png')

## Cell 14: Alignment Summary Report

In [ ]:
if not feedback_db:
    print('No feedback data yet.')
else:
    df = pd.DataFrame([asdict(f) for f in feedback_db])

    report_lines = [
        '# Multi-Agent Alignment Summary Report',
        f'**Session:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        f'**Total Feedback Records:** {len(feedback_db)}',
        f'**Total Preference Pairs (DPO):** {len(preference_db)}',
        '',
        '## Per-Agent Alignment Metrics',
        '',
    ]

    summary = df.groupby('agent_id').agg(
        avg_reasoning=('reasoning_score', 'mean'),
        avg_helpfulness=('helpfulness_score', 'mean'),
        avg_safety=('safety_score', 'mean'),
        avg_alignment=('alignment_score', 'mean'),
        feedback_count=('response_uid', 'count')
    ).round(2)

    report_lines.append(summary.to_markdown())
    report_lines.append('')
    report_lines.append('## Alignment Improvement Over Iterations')
    report_lines.append('')

    for res in all_pipeline_results:
        scores_p = [it['plan_feedback'].alignment_score for it in res['iterations']]
        scores_e = [it['exec_feedback'].alignment_score for it in res['iterations']]
        delta_p = scores_p[-1] - scores_p[0] if len(scores_p) > 1 else 0
        delta_e = scores_e[-1] - scores_e[0] if len(scores_e) > 1 else 0
        report_lines.append(f'**Task:** {res["task"][:60]}...')
        report_lines.append(f'- Planner alignment delta: **{delta_p:+.2f}**')
        report_lines.append(f'- Executor alignment delta: **{delta_e:+.2f}**')
        report_lines.append('')

    if preference_db:
        report_lines.append('## DPO Preference Pairs')
        report_lines.append(f'- Total pairs: {len(preference_db)}')
        avg_gap = np.mean([p.chosen_score - p.rejected_score for p in preference_db])
        report_lines.append(f'- Avg chosen-vs-rejected score gap: **{avg_gap:.2f}**')
        report_lines.append('')

    report_md = '\n'.join(report_lines)

    # Save to file
    with open('alignment_report.md', 'w') as f:
        f.write(report_md)

    display(Markdown(report_md))
    print('\n✅ Report saved to alignment_report.md')

## Cell 15: Export Feedback Database to CSV (for offline training)

In [ ]:
if feedback_db:
    df_feedback = pd.DataFrame([asdict(f) for f in feedback_db])
    df_feedback.to_csv('feedback_data.csv', index=False)
    print(f'✅ Feedback database exported: feedback_data.csv ({len(df_feedback)} rows)')
    print(df_feedback[['agent_id', 'reasoning_score', 'helpfulness_score',
                        'safety_score', 'alignment_score', 'critique']].to_string(index=False))

if preference_db:
    df_pref = pd.DataFrame([asdict(p) for p in preference_db])
    df_pref.to_csv('preference_pairs.csv', index=False)
    print(f'\n✅ Preference pairs exported: preference_pairs.csv ({len(df_pref)} rows)')

print('\n💡 These CSV files can be used to fine-tune models with RLHF / DPO pipelines.')